# Tutorial 8 — LLM Optimizer Settings: AdamW, Batch Size & Fine-Tuning

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part III — Pretraining**  
**Follows:** Tutorial 7 (Data Pipelines for Pretraining)  
**Precedes:** Tutorial 9 (Pretraining: Mixed Precision, Schedules & Scaling Laws)

---

## What This Tutorial Covers

[Tutorial 1b](../deep/tutorial_01b_optimizer.ipynb) derives all the optimizers from
scratch — GD, Momentum, RMSProp, Adam, AdamW, SGD — and includes the empirical
AdamW vs Adam+L2 comparison and batch size theory. This tutorial builds on
that foundation and focuses on **practical settings for language model
training**: why Adam's adaptive scaling is especially important for
transformer loss surfaces, LLM-specific hyperparameter defaults, token-level
batch size, gradient accumulation with mixed precision, and the hyperparameter
differences between pretraining and fine-tuning.

Topics:

1. **SGD and why it fails for transformers** — the ill-conditioned loss
   surface and why a global LR is insufficient.
2. **Adam** — LLM-specific defaults (`beta2=0.95`) and why they differ from
   the general-purpose defaults.
3. **AdamW** — decoupled weight decay; which parameters to weight-decay.
   (Empirical comparison in [Tutorial 1b](../deep/tutorial_01b_optimizer.ipynb).)
4. **Batch size** — recall from Tutorial 1b: critical batch size, linear
   scaling rule, generalization. Token-level batch size for language models.
5. **Learning rate** — LR range test implementation, practical LR table for
   pretraining and fine-tuning, catastrophic forgetting.
6. **Gradient accumulation** — correct implementation, the silent `zero_grad`
   bug, interaction with mixed precision (`torch.autocast`), and the
   variable-length sequences caveat.
7. **Pretraining vs fine-tuning settings** — full hyperparameter comparison
   table, layer-wise LR, and `OptimizerConfig` for production runs.

---

## 1. SGD — The Baseline

Stochastic gradient descent is the simplest update rule. At each step, take
a mini-batch of $B$ samples, compute the gradient of the loss with respect to
all parameters, and move in the negative gradient direction:

$$\theta_{t+1} = \theta_t - \eta \cdot g_t$$

where $g_t = \nabla_\theta \mathcal{L}(\theta_t; \mathcal{B}_t)$ is the
gradient on mini-batch $\mathcal{B}_t$ and $\eta$ is the learning rate.


### Why SGD fails for language models

The loss surface of a transformer is [**ill-conditioned**]{.underline}: gradients are
much larger in some directions than others. Consider a simple 2D example
with loss $\mathcal{L}(\theta_1, \theta_2) = \theta_1^2 + 100\,\theta_2^2$.
The gradient is $(2\theta_1, 200\theta_2)$ — the curvature in the
$\theta_2$ direction is 100× larger. Any learning rate large enough to make
progress in $\theta_1$ will cause oscillation in $\theta_2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate SGD on an ill-conditioned quadratic
theta = np.array([10.0, 1.0])   # starting point
lr    = 0.009                    # largest stable lr for θ₂

path_sgd = [theta.copy()]
for _ in range(100):
    grad  = np.array([2 * theta[0], 200 * theta[1]])
    theta = theta - lr * grad
    path_sgd.append(theta.copy())

path_sgd = np.array(path_sgd)

# Plot: notice the oscillation in the θ₂ direction
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(path_sgd[:, 0], label='θ₁')
axes[0].plot(path_sgd[:, 1], label='θ₂')
axes[0].set(title='SGD: parameter trajectories', xlabel='step')
axes[0].legend()

axes[1].plot(path_sgd[:, 0], path_sgd[:, 1], '-o', markersize=2)
axes[1].set(title='SGD: trajectory in parameter space',
            xlabel='θ₁', ylabel='θ₂')
plt.tight_layout(); plt.show()

The zigzagging is not just aesthetically unpleasant — it wastes gradient evaluations and slows convergence. In a 100M-parameter transformer, the condition number of the loss Hessian is orders of magnitude larger than 100. SGD simply cannot converge in reasonable time.

---

## 2. SGD with Momentum

Recall from [Tutorial 1b](../deep/tutorial_01b_optimizer.ipynb) that momentum replaces the raw gradient with an exponential moving average of gradient direction:

$$v_t = \mu \cdot v_{t-1} + g_t, \quad \theta_{t+1} = \theta_t - \eta \cdot v_t$$

Higher $\mu$ smooths cross-ravine oscillation; $\mu = 0.9$ is standard. PyTorch:
```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
```

### Why momentum still fails for transformers

Momentum addresses oscillation but not the core problem: [different parameters need different learning rates]{.mark}. Embedding parameters covering a large vocabulary need small updates; output projection parameters need larger ones early in training. A single global learning rate, even with momentum, cannot handle this.

The fix requires **per-parameter adaptive learning rates** — which is exactly what Adam provides.


---

## 3. Adam — Adaptive Moment Estimation

Recall from [Tutorial 1b](../deep/tutorial_01b_optimizer.ipynb) that Adam maintains per-parameter first (gradient direction) and second (gradient magnitude) moment estimates with bias correction:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}, \quad \theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

The denominator $\sqrt{\hat{v}_t}$ gives each parameter its own effective learning rate: small in high-curvature directions, large in flat ones. This is exactly what the ill-conditioned transformer surface requires.

### What the update rule actually does

- If $g_t$ has been **consistently large** (high $\hat{v}_t$): step size shrinks. The optimizer is cautious in high-curvature directions.
- If $g_t$ has been **small or noisy** (low $\hat{v}_t$): step size grows. The optimizer moves faster in flat, noisy directions.

The result: Adam effectively applies a **different local learning rate to each parameter** based on its gradient history.


### Default hyperparameters

| Hyperparameter | Default | What it controls |
|---|---|---|
| `lr` (η) | 1e-3 | Overall step size magnitude |
| `beta1` (β₁) | 0.9 | Momentum of gradient direction |
| `beta2` (β₂) | 0.999 | Momentum of gradient magnitude |
| `eps` (ε) | 1e-8 | Numerical stability |

[For language model training, `beta2=0.95` (GPT-3/NanoGPT default) is often preferred over `0.999`.]{.mark} The second moment "memory length" is $\approx 1/(1-\beta_2)$ steps: `0.999` retains ~1000 steps of gradient magnitude history while `0.95` retains only ~20. For long pretraining runs where gradient magnitudes shift significantly across phases (early chaos → stable convergence → late refinement), the shorter memory makes the adaptive scaling more responsive to recent behavior.


In [ ]:
# Standard Adam for language models
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),   # note: 0.95 not 0.999 for LLMs
    eps=1e-8,
)

---

## 4. AdamW — Fixing Weight Decay

As derived in [Tutorial 1b](../deep/tutorial_01b_optimizer.ipynb), adding L2 regularization to Adam's gradient is **not** the same as weight decay — the penalty gets divided by $\sqrt{\hat{v}_t}$, suppressing regularization exactly on the parameters that need it most (those with large gradient history). AdamW fixes this with a decoupled update applied **after** adaptive scaling:

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \eta \lambda \theta_t$$

[The weight decay term $\eta\lambda\theta_t$ is pure **shrinkage**]{.mark} — every parameter experiences the same regularization pressure, proportional only to its current magnitude. For the empirical comparison between Adam, Adam+L2, and AdamW, see [Tutorial 1b Section 6](../deep/tutorial_01b_optimizer.ipynb).


In [ ]:
# PyTorch AdamW
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,    # standard for LLM pretraining
)

### Which parameters to apply weight decay to

[Not all parameters should be weight-decayed:]{.underline}

- **Decayed:** weight matrices (`nn.Linear` weights, attention projections, FFN weights).
- **Not decayed:** bias terms, layer norm scale/shift parameters, embeddings — these are 1-D or serve as additive offsets; regularizing them hurts.


In [ ]:
def make_optimizer(model, lr, weight_decay, betas=(0.9, 0.95)):
    """Separate parameter groups: decay weights, don't decay biases/norms."""
    decay_params, no_decay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2:          # weight matrices
            decay_params.append(p)
        else:                    # biases, layer norm params (1-D tensors)
            no_decay_params.append(p)

    param_groups = [
        {'params': decay_params,    'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ]
    return torch.optim.AdamW(param_groups, lr=lr, betas=betas)

---

## 5. Batch Size for Language Models

Recall from [Tutorial 1b](../deep/tutorial_01b_optimizer.ipynb) Section 8:

- **Critical batch size** $B^* = \text{tr}(\boldsymbol{\Sigma})/\|\boldsymbol{g}_\text{true}\|^2$: the point of maximum gradient quality per FLOP. For LLM pretraining, $B^*$ falls in the range of 100K–500K tokens per step.
- **Linear scaling rule:** doubling $B$ → double $\eta$ (valid up to $B^*$).
- **Generalization:** smaller batches find flatter minima and generalize better; for fine-tuning on small datasets, batch sizes of 8–32 often outperform larger ones.

### Token-level batch size

For language models, the effective batch is always measured in **tokens** rather than samples, since sequences can have varying lengths:

$$\text{tokens per step} = \text{batch\_size} \times \text{seq\_len} \times \text{accum\_steps} \times n_\text{GPU}$$

Typical pretraining target: 256K–2M tokens per step (achieved via gradient accumulation when GPU memory limits the physical batch).


In [ ]:
# Linear scaling rule example
base_batch_size = 256
base_lr = 3e-4

new_batch_size = 1024
scaled_lr = base_lr * (new_batch_size / base_batch_size)
print(f"Scaled LR: {scaled_lr:.4f}")   # 1.2e-3

In [ ]:
# Token batch size calculation
batch_size   = 8          # samples per GPU
seq_len      = 1024       # tokens per sample
accum_steps  = 4          # gradient accumulation steps
num_gpus     = 1

tokens_per_step = batch_size * seq_len * accum_steps * num_gpus
print(f"Tokens per step: {tokens_per_step:,}")   # 32,768

---

## 6. Choosing a Learning Rate

The learning rate is the single most important hyperparameter. Too large
and the loss diverges; too small and the model trains too slowly or
settles in a poor minimum.

### The learning rate range test

The fastest way to find a reasonable LR: run a sweep from a very small
value (1e-7) to a large one (1e-1) over 100–200 steps, increasing the
LR exponentially. Plot the loss against the LR. The optimal LR is
slightly to the left of where the loss starts to diverge.

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
from copy import deepcopy

def lr_range_test(
    model,
    dataloader,
    criterion,
    min_lr: float = 1e-7,
    max_lr: float = 0.1,
    num_steps: int = 150,
):
    model_copy = deepcopy(model)
    optimizer  = torch.optim.AdamW(model_copy.parameters(), lr=min_lr,
                                   betas=(0.9, 0.95))
    lrs, losses = [], []

    ratio = (max_lr / min_lr) ** (1 / num_steps)

    data_iter = iter(dataloader)
    for step in range(num_steps):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            x, y = next(data_iter)

        optimizer.zero_grad()
        logits = model_copy(x)
        loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        optimizer.step()

        # Record before stepping LR
        current_lr = optimizer.param_groups[0]['lr']
        lrs.append(current_lr)
        losses.append(loss.item())

        # Exponentially increase LR
        for pg in optimizer.param_groups:
            pg['lr'] *= ratio

        if math.isnan(loss.item()) or loss.item() > 10 * losses[0]:
            break

    # Smooth losses with EMA for readability
    smoothed, alpha = [], 0.1
    running = losses[0]
    for l in losses:
        running = alpha * l + (1 - alpha) * running
        smoothed.append(running)

    plt.figure(figsize=(8, 4))
    plt.semilogx(lrs, smoothed)
    plt.xlabel('Learning Rate (log scale)')
    plt.ylabel('Loss (EMA smoothed)')
    plt.title('LR Range Test — pick LR just before the upturn')
    plt.grid(True, which='both', alpha=0.3)
    plt.show()

    return lrs, losses

### Practical LR ranges for LLMs

Based on empirical findings across many LLM training runs:

| Setting | Typical peak LR |
|---|---|
| Pretraining (125M–1B params) | 3e-4 – 6e-4 |
| Pretraining (7B+ params) | 1e-4 – 3e-4 |
| Full fine-tuning (any size) | 1e-5 – 5e-5 |
| LoRA fine-tuning (r=8–64) | 1e-4 – 3e-4 |
| DPO / GRPO | 1e-6 – 5e-6 |

The inverse relationship between model size and learning rate exists because
larger models have more redundancy — a given step size moves a smaller
fraction of the loss landscape. The fine-tuning values are 10–100× smaller
than pretraining because you are starting from a good point and want to
make small, targeted adjustments without destroying pretrained knowledge.

### Why fine-tuning LR matters more than pretraining LR

In pretraining, the model is randomly initialized and far from any
useful minimum. The loss landscape is rough. Large steps are necessary to
make progress, and the model is robust to LR missteps — you might slow
convergence, but you won't break anything fundamentally.

In fine-tuning:
- The pretrained weights encode vast knowledge. An LR that is too large
  performs **catastrophic forgetting**[^catforgetting] — gradient updates overwrite learned
  representations.
- The fine-tuning dataset is small. Large LR causes **overfitting to noise**
  in a few steps.
- With LoRA, the base weights are frozen, but the adapter weights ($A$, $B$)
  start at magnitude ~0. A large LR makes them grow too fast relative to
  the base forward pass.

[^catforgetting]: Catastrophic forgetting occurs when fine-tuning gradient updates are large enough to overwrite the distributed representations learned during pretraining. The pretrained weights encode general language knowledge across all parameters simultaneously — a large update to any one pushes the weight away from the entire knowledge manifold, not just the task-specific part.

In [ ]:
# Catastrophic forgetting demo: compare fine-tuning LR
# Pretrain a small model, then fine-tune with two different LRs

def measure_catastrophic_forgetting(pretrained_model, finetune_data,
                                    eval_data_pretrain, lr):
    """Returns: (finetune_loss_after, pretrain_skill_retention)."""
    model = deepcopy(pretrained_model)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    for x, y in finetune_data:          # just a few steps
        opt.zero_grad()
        logits = model(x)
        loss   = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)),
                                             y.view(-1))
        loss.backward()
        opt.step()

    with torch.no_grad():
        retention_losses = []
        for x, y in eval_data_pretrain:
            logits = model(x)
            retention_losses.append(
                nn.functional.cross_entropy(logits.view(-1, logits.size(-1)),
                                            y.view(-1)).item()
            )
    return sum(retention_losses) / len(retention_losses)

---

## 7. Gradient Accumulation

### The problem: effective batch size vs GPU memory

You want a token batch size of 512K tokens (required for stable pretraining
at moderate scale). Your GPU can hold at most 8 sequences of 1024 tokens in
memory — 8,192 tokens. The gap is 64×.

Gradient accumulation solves this by splitting the effective batch into
$k$ **micro-batches**, running a forward/backward pass on each, and only
calling `optimizer.step()` after all $k$ micro-batches. The gradients
accumulate (add) across micro-batches before the update.

$$\text{effective\_batch\_size} = \text{micro\_batch\_size} \times k \times n_{\text{GPU}}$$

Since gradients are additive ($\nabla(\sum_i L_i) = \sum_i \nabla L_i$),
this is mathematically equivalent to a single forward pass over the full
batch — as long as you divide the loss by $k$.

### Why you must divide by $k$

If you do not divide by $k$, the gradient scale grows linearly with the
accumulation steps. After $k$ micro-steps, the accumulated gradient is
$k$ times larger than the single-step gradient, making the effective
learning rate $k\eta$ instead of $\eta$. At large $k$ (e.g., $k=32$),
this causes immediate loss divergence.

$$\text{loss per micro-step} = \frac{L(\mathcal{B}_i)}{k}$$

In [ ]:
# Correct gradient accumulation
def train_step_with_accumulation(model, optimizer, get_batch,
                                 accum_steps: int):
    optimizer.zero_grad()                    # zero ONCE before the loop
    total_loss = 0.0

    for micro_step in range(accum_steps):
        x, y = get_batch()
        logits, loss = model(x, y)

        # Scale the loss by 1/k before backward
        (loss / accum_steps).backward()      # gradients accumulate in .grad

        total_loss += loss.item()

    # After all micro-steps, gradients represent the full effective batch
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return total_loss / accum_steps

### [The silent `zero_grad` bug]{.underline}

If you call `optimizer.zero_grad()` inside the accumulation loop
(once per micro-step), gradients are wiped between micro-steps. Only the
last micro-step's gradient survives into the `optimizer.step()`:

In [ ]:
# BUG: zero_grad inside the loop
for micro_step in range(accum_steps):
    optimizer.zero_grad()   # ← WRONG: discards all previous accumulation
    x, y = get_batch()
    _, loss = model(x, y)
    (loss / accum_steps).backward()
optimizer.step()
# Result: equivalent to batch_size, not batch_size * accum_steps
# The loss will look fine — this bug is silent

### Gradient accumulation with mixed precision

When using `torch.autocast`, place the context manager inside the
accumulation loop (around the forward pass), not around the full loop:

In [ ]:
# Correct: autocast per micro-step
optimizer.zero_grad()
for micro_step in range(accum_steps):
    x, y = get_batch()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = model(x, y)
    (loss / accum_steps).backward()   # backward outside autocast is fine

optimizer.step()

This is correct because `torch.autocast` only affects the forward pass. The backward pass and optimizer step always run in FP32.

:::{dropdown} ⓘ &nbsp; Gradient accumulation with variable-length sequences
For language models, cross-entropy is typically averaged over *tokens*, not sequences. The number of tokens per micro-batch varies (different sequence lengths, masked positions). Dividing by $S$ is then **not** equivalent to the full-batch gradient — correct normalization requires dividing by the total token count $N = \sum_a N_a$ after all micro-batches:

```python
IGNORE_IDX = -100
m = 0
for i, (x, y) in enumerate(train_loader):
    outs = model(x)
    loss = F.cross_entropy(outs, y, reduction="sum")
    loss.backward()
    m += (y != IGNORE_IDX).int().sum()

    if (i + 1) % S == 0:
        for p in model.parameters():
            p.grad /= m           # normalize by true token count
        optimizer.step()
        optimizer.zero_grad()
        m = 0
```

Using a fixed divisor $S$ when token counts vary creates a silent scaling error proportional to the average length mismatch. See this [blog post](https://unsloth.ai/blog/gradient) for a detailed discussion of the downstream training pathologies this causes.
:::


---

## 8. Optimizer Hyperparameters: Pretraining vs Fine-Tuning

The settings that work for pretraining do not directly transfer to
fine-tuning. Here is a concrete comparison:

| Hyperparameter | Pretraining | Fine-tuning (Full) | Fine-tuning (LoRA) |
|---|---|---|---|
| Optimizer | AdamW | AdamW | AdamW |
| Peak LR | 3e-4 (125M), 1e-4 (1B+) | 1e-5 – 5e-5 | 1e-4 – 3e-4 |
| β₁ | 0.9 | 0.9 | 0.9 |
| β₂ | 0.95 | 0.999 | 0.999 |
| Weight decay | 0.1 | 0.01 – 0.1 | 0.01 |
| Warmup steps | 1–2% of total | 3–10% of total | 10% of total |
| Grad clip | 1.0 | 1.0 | 1.0 |
| Effective batch (tokens) | 256K – 4M | 8K – 128K | 4K – 32K |
| Accum steps | 4–64 | 1–8 | 1–4 |

### Why β₂ changes for fine-tuning

In pretraining, `beta2=0.95` means the second moment forgets the past after
roughly 20 steps. This makes adaptive scaling responsive to rapid gradient
changes across a long run of millions of steps.

In fine-tuning, runs are shorter (hundreds to thousands of steps) and gradient
magnitudes change less dramatically (you are near a good minimum). `beta2=0.999`
provides smoother, more stable second-moment estimates, which avoids noisy
adaptive scaling that could destabilize the fine-tune.

### Why weight decay is lower for fine-tuning

High weight decay during fine-tuning fights the gradient update: the model
tries to adapt to the new task while decay constantly pulls weights toward
zero. For full fine-tuning, `weight_decay=0.01` is common. For LoRA (where
adapter weights start at ~0), even `0.01` can slow convergence — some
practitioners use `0.0` for LoRA adapters.

### Parameter groups in fine-tuning: different LR for different layers

For full fine-tuning, early layers (general features) benefit from a smaller
LR than later layers (task-specific representations):


In [ ]:
def make_layerwise_optimizer(model, base_lr, num_layers):
    """
    Linear LR decay: layer 0 gets base_lr * 0.1, last layer gets base_lr.
    Forces early layers to stay close to pretrained representations.
    """
    param_groups = []
    for i, (name, p) in enumerate(model.named_parameters()):
        # Estimate layer depth from parameter name
        layer_idx = 0
        for j in range(num_layers):
            if f'layers.{j}.' in name or f'layer.{j}.' in name:
                layer_idx = j + 1
                break

        lr_scale = 0.1 + 0.9 * (layer_idx / num_layers)   # 0.1 → 1.0
        param_groups.append({
            'params': [p],
            'lr':     base_lr * lr_scale,
            'weight_decay': 0.01 if p.ndim >= 2 else 0.0,
        })

    return torch.optim.AdamW(param_groups, lr=base_lr, betas=(0.9, 0.999))

---

## 9. A Complete Optimizer Setup for Fine-Tuning

Bringing all of the above together into a reusable fine-tuning optimizer
configuration:

In [ ]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR
import math
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class OptimizerConfig:
    # Core
    lr:             float = 2e-5
    weight_decay:   float = 0.01
    betas:          tuple = (0.9, 0.999)
    eps:            float = 1e-8
    grad_clip:      float = 1.0

    # Schedule
    warmup_ratio:   float = 0.06      # fraction of steps for warmup
    min_lr_ratio:   float = 0.1       # min_lr = lr * min_lr_ratio

    # Accumulation
    accum_steps:    int   = 1

    # LoRA-specific
    lora_lr_scale:  float = 1.0       # LoRA params can use higher LR if needed


def build_optimizer(model: nn.Module, config: OptimizerConfig,
                    total_steps: int):
    """
    Returns (optimizer, scheduler) ready for a fine-tuning run.
    """
    # Split parameter groups: decay weights, skip biases + norm params
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2:
            decay.append(p)
        else:
            no_decay.append(p)

    param_groups = [
        {'params': decay,    'weight_decay': config.weight_decay},
        {'params': no_decay, 'weight_decay': 0.0},
    ]

    optimizer = torch.optim.AdamW(
        param_groups,
        lr=config.lr,
        betas=config.betas,
        eps=config.eps,
    )

    # Cosine schedule with warmup
    warmup_steps = int(total_steps * config.warmup_ratio)
    min_ratio    = config.min_lr_ratio

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        progress = min(progress, 1.0)
        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def fine_tune_step(model, optimizer, scheduler, get_micro_batch,
                   accum_steps: int, grad_clip: float) -> float:
    """One optimizer step with gradient accumulation."""
    optimizer.zero_grad()
    total_loss = 0.0

    for _ in range(accum_steps):
        x, y = get_micro_batch()
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model(x, y)
        (loss / accum_steps).backward()
        total_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    scheduler.step()

    return total_loss / accum_steps

---

## Summary

| Concept | Key point |
|---|---|
| SGD for LLMs | Ill-conditioned transformer surfaces require per-parameter LR; SGD with a single global LR cannot converge at scale. |
| Adam | Per-parameter adaptive LR; LLM default: `beta2=0.95` (memory ~20 steps) vs general `0.999` (memory ~1000 steps). |
| AdamW | Decoupled weight decay = uniform shrinkage regardless of gradient history. Always prefer over Adam+L2. |
| Weight decay | Decay weight matrices (`ndim≥2`); skip biases, layer norm parameters, embeddings. |
| Token batch size | tokens/step = batch_size × seq_len × accum_steps × n_GPU; target 256K–2M for pretraining. |
| Fine-tuning LR | 10–100× smaller than pretraining; catastrophic forgetting if too large. |
| Gradient accumulation | Divide loss by $k$; `zero_grad` once before the loop; wrap `autocast` around each micro-step. |
| Variable-length sequences | Divide by total token count (not $k$) when sequence lengths vary across micro-batches. |
| β₂ for fine-tuning | Use `0.999` (not `0.95`); shorter runs and stable gradient magnitudes warrant longer memory. |
| Pretraining vs fine-tuning | Lower LR, higher β₂, lower weight decay, more warmup, smaller effective batch for fine-tuning. |
